# Stage 3 Wander — Boundary and Obstacle Avoidance

Camera tilts down at startup so it sees more floor.  
Robot drives forward (`WANDER`) and reacts to two hazards:
- **Yellow tape** detected below row 65 % → `AVOID_YELLOW`
- **Canny-edge density** in the front-middle ROI → `AVOID_OBSTACLE`

No cap detection, no arm control, no Roboflow.

States: `WANDER` → `AVOID_YELLOW` / `AVOID_OBSTACLE` → `WANDER` → …

## 1. Imports

In [ ]:
import cv2
import numpy as np
import time
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg

print('Imports OK')

In [ ]:
try:
    from SCSCtrl import TTLServo
    _servo_available = True
    print('TTLServo loaded.')
except Exception as _e:
    _servo_available = False
    print('TTLServo not available:', _e)
    print('Camera tilt will be skipped.')

## 2. Configuration

Tune `YELLOW_BOUNDARY_THRESHOLD` and `OBSTACLE_EDGE_THRESHOLD` using **Test 5** before running the full wander.  
`CAMERA_TILT_DOWN` — degrees to tilt camera downward. The colorTracking notebook uses 25 as max down; 15 is a safe default.

In [ ]:
# ── Camera ──────────────────────────────────────────────────────────
CAMERA_WIDTH  = 300
CAMERA_HEIGHT = 300

# ── Camera tilt ──────────────────────────────────────────────────────
# Servo 1 = PAN (horizontal).  Servo 5 = TILT (up/down).
# 0 = horizontal, positive = tilt down, max ~25 from colorTracking example.
CAMERA_TILT_DOWN = 15   # degrees; safe range 0-25

# ── Motor speeds ─────────────────────────────────────────────────────
FORWARD_SPEED = 0.18
REVERSE_SPEED = 0.15
TURN_SPEED    = 0.22
MAX_SPEED     = 0.25

# ── Avoidance timing ─────────────────────────────────────────────────
AVOID_REVERSE_TIME = 0.40   # seconds to reverse
AVOID_TURN_TIME    = 0.55   # seconds to turn
WANDER_COOLDOWN    = 0.50   # free drive after avoidance before re-checking

# ── Yellow tape ROI ──────────────────────────────────────────────────
# Yellow tape is only checked below this row fraction.
# 0.65 means the bottom 35 % of the frame (rows 65-100 %).
YELLOW_ROI_START = 0.65

# ── Obstacle ROI ─────────────────────────────────────────────────────
FRONT_ROI_TOP_FRAC    = 0.15
FRONT_ROI_BOTTOM_FRAC = 0.60

# ── Detection thresholds ─────────────────────────────────────────────
YELLOW_BOUNDARY_THRESHOLD = 1800
OBSTACLE_EDGE_THRESHOLD   = 500
OBSTACLE_EQUAL_THRESH     = 50   # left/right diff below this = equal sides

# ── HSV for yellow tape ───────────────────────────────────────────────
TAPE_HSV_LOWER = np.array([15,  60,  60])
TAPE_HSV_UPPER = np.array([45, 255, 255])

# ── Canny ─────────────────────────────────────────────────────────────
CANNY_LOW  = 40
CANNY_HIGH = 120

print('Configuration loaded.')
print('  Camera tilt down:', CAMERA_TILT_DOWN, 'deg (servo 5)')
print('  Yellow ROI start:', YELLOW_ROI_START,
      '-> bottom', round((1.0 - YELLOW_ROI_START) * 100), '% of frame')
print('  Thresholds — yellow:', YELLOW_BOUNDARY_THRESHOLD,
      ' obstacle:', OBSTACLE_EDGE_THRESHOLD)

## 3. Camera Tilt

**Servo 1** = PAN (horizontal left/right).  
**Servo 5** = TILT (up/down) — this is what we adjust to see more floor.  
From colorTracking: `servoAngleCtrl(5, 25, 1, speed)` is max down.  
We use `CAMERA_TILT_DOWN = 15` as a conservative default.

In [ ]:
if _servo_available:
    try:
        TTLServo.servoAngleCtrl(1, 0, 1, 100)                      # center pan
        time.sleep(0.15)
        TTLServo.servoAngleCtrl(5, CAMERA_TILT_DOWN, 1, 100)       # tilt down
        time.sleep(0.50)
        print('Camera: pan centered, tilt down', CAMERA_TILT_DOWN, 'deg')
    except Exception as e:
        print('Servo init error:', e)
else:
    print('Servo not available — camera tilt skipped.')

## 4. Camera and Robot

Camera is a singleton — safe to re-run if it already exists.  
If the camera fails to open, run **Kernel → Shutdown All Kernels** first to release it.

In [ ]:
camera = Camera.instance(width=CAMERA_WIDTH, height=CAMERA_HEIGHT)

image_widget = widgets.Image(format='jpeg', width=CAMERA_WIDTH,  height=CAMERA_HEIGHT)
mask_widget  = widgets.Image(format='jpeg', width=CAMERA_WIDTH,  height=CAMERA_HEIGHT)
state_label  = widgets.Label(value='State: STOPPED')
motor_label  = widgets.Label(value='Motors: L=0.00  R=0.00')
yellow_label = widgets.Label(value='Yellow: 0  (L=0 R=0)')
obs_label    = widgets.Label(value='Edges: 0  (L=0 R=0)')
turn_label   = widgets.Label(value='Turn dir: -')

display(widgets.VBox([
    widgets.HBox([image_widget, mask_widget]),
    widgets.VBox([state_label, motor_label,
                  yellow_label, obs_label, turn_label])
]))
print('Camera ready.')

In [ ]:
robot = Robot()
robot.stop()
print('Robot initialized and stopped.')

## 5. Helper Functions

In [ ]:
def clamp(val, lo, hi):
    if val < lo: return lo
    if val > hi: return hi
    return val


def safe_stop():
    try:
        robot.left_motor.value  = 0.0
        robot.right_motor.value = 0.0
    except Exception:
        pass


def set_drive(left, right):
    robot.left_motor.value  = clamp(left,  -MAX_SPEED, MAX_SPEED)
    robot.right_motor.value = clamp(right, -MAX_SPEED, MAX_SPEED)


print('clamp / safe_stop / set_drive defined.')

## 6. Yellow Boundary Detection

Only checks frame rows **below `YELLOW_ROI_START`** (default 65 % → bottom 35 %).  
Yellow tape higher in the frame (sky, objects) is ignored.

In [ ]:
def detect_yellow_boundary(frame):
    h         = frame.shape[0]
    roi_start = int(h * YELLOW_ROI_START)
    roi       = frame[roi_start:, :]

    hsv_roi  = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    roi_mask = cv2.inRange(hsv_roi, TAPE_HSV_LOWER, TAPE_HSV_UPPER)

    yellow_area  = int(np.sum(roi_mask > 0))
    mid          = roi_mask.shape[1] // 2
    yellow_left  = int(np.sum(roi_mask[:, :mid] > 0))
    yellow_right = int(np.sum(roi_mask[:, mid:] > 0))

    full_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    full_mask[roi_start:, :] = roi_mask

    return {
        'detected':     yellow_area >= YELLOW_BOUNDARY_THRESHOLD,
        'yellow_area':  yellow_area,
        'yellow_left':  yellow_left,
        'yellow_right': yellow_right,
        'yellow_mask':  full_mask,
    }


print('detect_yellow_boundary() defined.')

## 7. Obstacle Detection

Checks rows **15 – 60 %** of frame height (front-middle strip).  
Yellow pixels are masked out first so floor tape does not count.  

Turn direction:
- More edges on **left** → turn **right** (empty side)
- More edges on **right** → turn **left** (empty side)
- Roughly **equal** → **alternate** left/right each time

In [ ]:
def detect_obstacle(frame):
    h, w    = frame.shape[:2]
    roi_top = int(h * FRONT_ROI_TOP_FRAC)
    roi_bot = int(h * FRONT_ROI_BOTTOM_FRAC)
    roi     = frame[roi_top:roi_bot, :]

    # Mask yellow pixels so floor tape does not register as obstacle
    hsv_roi     = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    yellow_mask = cv2.inRange(hsv_roi, TAPE_HSV_LOWER, TAPE_HSV_UPPER)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray[yellow_mask > 0] = 0

    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges   = cv2.Canny(blurred, CANNY_LOW, CANNY_HIGH)
    edges   = cv2.dilate(edges, None, iterations=1)

    mid        = edges.shape[1] // 2
    edge_left  = int(np.sum(edges[:, :mid] > 0))
    edge_right = int(np.sum(edges[:, mid:] > 0))
    edge_total = edge_left + edge_right

    full_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    full_mask[roi_top:roi_bot, :] = edges

    return {
        'detected':   edge_total >= OBSTACLE_EDGE_THRESHOLD,
        'edge_total': edge_total,
        'edge_left':  edge_left,
        'edge_right': edge_right,
        'mask':       full_mask,
    }


print('detect_obstacle() defined.')

## 8. Debug Visualization

Left widget (annotated frame):
- **Cyan horizontal line** at `YELLOW_ROI_START` — yellow detection begins here
- **Orange rectangle** — obstacle detection zone (turns red when firing)
- Text: state `S`, motors `L/R`, yellow area `Y`, edge count `E`, turn dir `T`

Right widget: **cyan** = yellow tape pixels, **orange** = obstacle edge pixels

In [ ]:
def draw_debug(frame, boundary, obstacle, state, left_spd, right_spd, turn_dir):
    out  = frame.copy()
    h, w = out.shape[:2]

    # Yellow ROI: horizontal line + light tint below it
    roi_y   = int(h * YELLOW_ROI_START)
    y_color = (0, 0, 255) if boundary['detected'] else (0, 220, 220)
    overlay = out.copy()
    cv2.rectangle(overlay, (0, roi_y), (w - 1, h - 1), y_color, -1)
    cv2.addWeighted(overlay, 0.12, out, 0.88, 0, out)
    cv2.line(out, (0, roi_y), (w - 1, roi_y), y_color, 2)

    # Obstacle ROI rectangle
    obs_top = int(h * FRONT_ROI_TOP_FRAC)
    obs_bot = int(h * FRONT_ROI_BOTTOM_FRAC)
    o_color = (0, 0, 255) if obstacle['detected'] else (255, 130, 0)
    cv2.rectangle(out, (0, obs_top), (w - 1, obs_bot), o_color, 2)

    # Vertical centerline
    cv2.line(out, (w // 2, 0), (w // 2, h), (70, 70, 70), 1)

    # Text overlay
    cv2.putText(out, 'S:' + state,
                (5, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1)
    cv2.putText(out,
                'L=' + str(round(left_spd, 2)) + ' R=' + str(round(right_spd, 2)),
                (5, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)
    cv2.putText(out, 'Y=' + str(boundary['yellow_area']),
                (5, 49), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 240, 240), 1)
    cv2.putText(out, 'E=' + str(obstacle['edge_total']),
                (5, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255, 130, 0), 1)
    cv2.putText(out, 'T:' + turn_dir,
                (5, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (200, 200, 0), 1)

    if boundary['detected']:
        cv2.putText(out, 'YELLOW!',
                    (5, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (0, 0, 255), 2)
    elif obstacle['detected']:
        cv2.putText(out, 'OBSTACLE!',
                    (5, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (0, 80, 255), 2)

    return out


print('draw_debug() defined.')

## 9. State Machine

| State | Behaviour |
|-------|-----------|
| `WANDER` | Drive forward. Check yellow first (higher priority), then obstacle. |
| `AVOID_YELLOW` | Phase 1: reverse. Phase 2: turn away from yellow-heavy side. Phase 3: `WANDER`. |
| `AVOID_OBSTACLE` | Phase 1: reverse. Phase 2: turn toward the emptier side. Phase 3: `WANDER`. |
| `STOPPED` | Emergency only. No auto-recovery. |

Avoidance uses timestamp-based phases — no blocking `sleep()` inside the callback.

In [ ]:
nav_state           = 'WANDER'
avoid_start_time    = 0.0
avoid_turn_right    = True
_obstacle_alt_right = True   # flips each time obstacle sides are equal
wander_cooldown_end = 0.0
last_left_spd       = 0.0
last_right_spd      = 0.0

print('State machine ready.  nav_state =', nav_state)

In [ ]:
def execute(change):
    global nav_state, avoid_start_time, avoid_turn_right
    global _obstacle_alt_right, wander_cooldown_end
    global last_left_spd, last_right_spd

    frame     = change['new']
    left_spd  = 0.0
    right_spd = 0.0

    try:
        boundary = detect_yellow_boundary(frame)
        obstacle = detect_obstacle(frame)
        now      = time.time()

        # ── State machine ────────────────────────────────────────

        if nav_state == 'STOPPED':
            pass   # only manual reset can leave STOPPED

        elif nav_state in ('AVOID_YELLOW', 'AVOID_OBSTACLE'):
            elapsed = now - avoid_start_time
            if elapsed < AVOID_REVERSE_TIME:
                # Phase 1: reverse straight back
                left_spd  = -REVERSE_SPEED
                right_spd = -REVERSE_SPEED
            elif elapsed < AVOID_REVERSE_TIME + AVOID_TURN_TIME:
                # Phase 2: turn toward the free side
                if avoid_turn_right:
                    left_spd  =  TURN_SPEED
                    right_spd = -TURN_SPEED
                else:
                    left_spd  = -TURN_SPEED
                    right_spd =  TURN_SPEED
            else:
                # Phase 3: resume wander with cooldown
                wander_cooldown_end = now + WANDER_COOLDOWN
                nav_state = 'WANDER'
                left_spd  = FORWARD_SPEED
                right_spd = FORWARD_SPEED

        elif nav_state == 'WANDER':
            if now < wander_cooldown_end:
                # Post-avoidance free run — skip detection
                left_spd  = FORWARD_SPEED
                right_spd = FORWARD_SPEED

            elif boundary['detected']:
                # Yellow tape: turn away from the yellower half
                avoid_turn_right = boundary['yellow_left'] >= boundary['yellow_right']
                avoid_start_time = now
                nav_state        = 'AVOID_YELLOW'
                left_spd         = -REVERSE_SPEED
                right_spd        = -REVERSE_SPEED

            elif obstacle['detected']:
                # Obstacle: turn toward the emptier half
                el = obstacle['edge_left']
                er = obstacle['edge_right']
                if el - er > OBSTACLE_EQUAL_THRESH:
                    avoid_turn_right = True    # more on left  → turn right
                elif er - el > OBSTACLE_EQUAL_THRESH:
                    avoid_turn_right = False   # more on right → turn left
                else:
                    # Equal sides — alternate to avoid getting stuck
                    avoid_turn_right    = _obstacle_alt_right
                    _obstacle_alt_right = not _obstacle_alt_right
                avoid_start_time = now
                nav_state        = 'AVOID_OBSTACLE'
                left_spd         = -REVERSE_SPEED
                right_spd        = -REVERSE_SPEED

            else:
                # Clear path — drive forward
                left_spd  = FORWARD_SPEED
                right_spd = FORWARD_SPEED

        # ── Drive ────────────────────────────────────────────────
        set_drive(left_spd, right_spd)
        last_left_spd  = left_spd
        last_right_spd = right_spd

        # ── Debug display ────────────────────────────────────────
        td                 = 'RIGHT' if avoid_turn_right else 'LEFT'
        annotated          = draw_debug(frame, boundary, obstacle,
                                        nav_state, left_spd, right_spd, td)
        image_widget.value = bgr8_to_jpeg(annotated)

        # Cyan = yellow tape pixels, orange = obstacle edge pixels
        combined = np.zeros((CAMERA_HEIGHT, CAMERA_WIDTH, 3), dtype=np.uint8)
        ym = boundary['yellow_mask']
        om = obstacle['mask']
        combined[ym > 0] = [0, 255, 255]
        combined[om > 0] = [0, 140, 255]
        mask_widget.value = bgr8_to_jpeg(combined)

        state_label.value  = 'State: ' + nav_state
        motor_label.value  = ('Motors L=' + str(round(left_spd, 3)) +
                              '  R=' + str(round(right_spd, 3)))
        yellow_label.value = ('Yellow: ' + str(boundary['yellow_area']) +
                              '  L=' + str(boundary['yellow_left']) +
                              '  R=' + str(boundary['yellow_right']))
        obs_label.value    = ('Edges: ' + str(obstacle['edge_total']) +
                              '  L=' + str(obstacle['edge_left']) +
                              '  R=' + str(obstacle['edge_right']))
        turn_label.value   = 'Turn dir: ' + td

    except Exception as e:
        safe_stop()
        print('[ERROR] ' + str(e))
        raise


print('execute() defined.')

## 10. Start Navigation

Run this cell to start. **Robot moves immediately after `camera.observe()` is called.**

In [ ]:
nav_state           = 'WANDER'
avoid_start_time    = 0.0
avoid_turn_right    = True
_obstacle_alt_right = True
wander_cooldown_end = 0.0
last_left_spd       = 0.0
last_right_spd      = 0.0

camera.unobserve_all()
camera.observe(execute, names='value')
print('Navigation started.  State:', nav_state)

## 11. Emergency Stop

Run immediately if the robot misbehaves.

In [ ]:
camera.unobserve_all()
robot.stop()
nav_state = 'STOPPED'
running   = False
state_label.value = 'State: STOPPED'
motor_label.value = 'Motors: L=0.00  R=0.00'
print('EMERGENCY STOP — camera detached, motors zeroed.')

## 12. Reset and Restart

After stopping, reset state and restart without re-running all cells.

In [ ]:
camera.unobserve_all()
robot.stop()

nav_state           = 'WANDER'
avoid_start_time    = 0.0
avoid_turn_right    = True
_obstacle_alt_right = True
wander_cooldown_end = 0.0
last_left_spd       = 0.0
last_right_spd      = 0.0

camera.observe(execute, names='value')
print('Reset to WANDER.  Navigation restarted.')

## 13. Test Procedures

Run these **before** starting the full wander to verify each subsystem.

---

### Test 1 — Yellow Boundary Detection (static)

Hold yellow tape below the camera or position the robot near the arena edge.

```python
frame  = camera.value.copy()
result = detect_yellow_boundary(frame)
print('detected:', result['detected'],
      ' area:', result['yellow_area'],
      ' L:', result['yellow_left'],
      ' R:', result['yellow_right'])

vis   = frame.copy()
h     = vis.shape[0]
roi_y = int(h * YELLOW_ROI_START)
cv2.line(vis, (0, roi_y), (vis.shape[1]-1, roi_y), (0, 220, 220), 2)
mask_bgr = cv2.cvtColor(result['yellow_mask'], cv2.COLOR_GRAY2BGR)
display(widgets.Image(value=bgr8_to_jpeg(vis),      format='jpeg'))
display(widgets.Image(value=bgr8_to_jpeg(mask_bgr), format='jpeg'))
```

**Pass:** `detected` is `True` with tape visible, `False` on clean floor.  
Adjust `YELLOW_BOUNDARY_THRESHOLD` if needed.

---

### Test 2 — Obstacle Detection (static)

Hold a cardboard box ~30 cm in front of the robot.

```python
frame  = camera.value.copy()
result = detect_obstacle(frame)
print('detected:', result['detected'],
      ' edges:', result['edge_total'],
      ' L:', result['edge_left'],
      ' R:', result['edge_right'])

edge_bgr = cv2.cvtColor(result['mask'], cv2.COLOR_GRAY2BGR)
display(widgets.Image(value=bgr8_to_jpeg(edge_bgr), format='jpeg'))
```

**Pass:** `detected` is `True` with box present, `False` on clear floor.  
Adjust `OBSTACLE_EDGE_THRESHOLD` or `CANNY_LOW/HIGH` if floor generates false positives.

---

### Test 3 — Motor Directions (robot on blocks)

Lift the robot so tracks spin freely.

```python
import time
print('Forward 1 s')
set_drive(FORWARD_SPEED,  FORWARD_SPEED);  time.sleep(1.0); safe_stop()
time.sleep(0.5)
print('Reverse 1 s')
set_drive(-REVERSE_SPEED, -REVERSE_SPEED); time.sleep(1.0); safe_stop()
time.sleep(0.5)
print('Turn right 1 s')
set_drive(TURN_SPEED,     -TURN_SPEED);    time.sleep(1.0); safe_stop()
```

**Pass:** tracks spin in the expected directions.

---

### Test 4 — Full Wander (robot in arena)

Run **Section 10 — Start Navigation**.  

**Expected:**
1. Robot drives forward slowly in `WANDER`.
2. Yellow tape in bottom ROI → `AVOID_YELLOW`: reverse, turn, back to `WANDER`.
3. Box in front ROI → `AVOID_OBSTACLE`: reverse, turn toward empty side, back to `WANDER`.
4. Cycle repeats indefinitely.

If the robot turns the wrong way on an obstacle, check the `edge_left` / `edge_right` values  
printed in `obs_label` and adjust `OBSTACLE_EQUAL_THRESH`.

---

### Test 5 — Threshold Calibration (stationary, clear arena)

```python
frame = camera.value.copy()
b = detect_yellow_boundary(frame)
o = detect_obstacle(frame)
print('Baseline — yellow:', b['yellow_area'], '  edges:', o['edge_total'])
```

Set `YELLOW_BOUNDARY_THRESHOLD` ≈ **3× baseline yellow area**.  
Set `OBSTACLE_EDGE_THRESHOLD`   ≈ **3× baseline edge count**.